# 03 - Calibrate Threshold (3-slice v2, ResNet-50)

Calibre le seuil ResNet-50 sur la val avec le contexte 3-slice + crop=30. Écrit `optimal_threshold_resnet50_wide_crop.{txt,json}`.

In [ ]:
#@title Colab setup
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/moradBMH/INF8225_Projet.git"
GIT_REF = "clean-structure"
PROJECT_DIR = Path("/content/INF8225_Projet")
DRIVE_FOLDER = None
INSTALL_DEPS = True
REINSTALL_DEPS = False

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=False)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from colab.setup import setup
setup(drive_folder=DRIVE_FOLDER, install=INSTALL_DEPS, reinstall=REINSTALL_DEPS)
print("cwd:", Path.cwd())

In [ ]:
#@title Verify 3-slice v2 checkpoints are present
from pathlib import Path
from msd_recall_strategy import get_resnet_checkpoint_dir

checkpoint_dir = get_resnet_checkpoint_dir()
required = [
    Path("data/MSD_pancreas/val.json"),
    Path("work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth"),
    Path("msd_implementation/configs/grounding_dino/pancreas_tumor.py"),
]
required += [checkpoint_dir / f"resnet50_wide_crop_fold_{i}.pth" for i in range(1, 6)]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing required files: {missing}. Lance d'abord les notebooks 01 et 02 v2."

In [ ]:
#@title Run pipeline step (3-slice v2 calibration)
import subprocess
import sys
subprocess.run([sys.executable, "-u", "-m", "msd_implementation.pipelines.resnet50_wide_crop.calibrate_threshold"], check=True)

In [ ]:
#@title Inspect 3-slice v2 threshold artifacts
from pathlib import Path
for p in [Path("outputs/msd_implementation/resnet50_wide_crop/metrics/optimal_threshold_resnet50_wide_crop.txt"), Path("outputs/msd_implementation/resnet50_wide_crop/metrics/optimal_threshold_resnet50_wide_crop.json"), Path("outputs/msd_implementation/resnet50_wide_crop/metrics/calibration_threshold_resnet50_wide_crop.csv"), Path("outputs/msd_implementation/resnet50_wide_crop/metrics/threshold_sweep_3slice_v2.csv")]:
    print(("OK     " if p.exists() else "MISSING"), p)
assert Path("outputs/msd_implementation/resnet50_wide_crop/metrics/optimal_threshold_resnet50_wide_crop.txt").exists(), "outputs/msd_implementation/resnet50_wide_crop/metrics/optimal_threshold_resnet50_wide_crop.txt missing"
print("threshold:", Path("outputs/msd_implementation/resnet50_wide_crop/metrics/optimal_threshold_resnet50_wide_crop.txt").read_text().strip())